# 개선 루프

* 지속적 개선은 단일 메커니즘이 아니라 피드백 파이프라인을 사용해 이슈를 진단하고 실험을 수행하며 학습하는 상호연결된 사이클
* 지속적 개선은 에이전트가 환경과 반복 상호작용하며 최적 행동을 학습하는 방식

* 개선 루프 방법

    | 기법 | 목적 | 강점 | 한계 | 사용 시점 |
    |-----|-----|-----|-----|-----|
    | 피드백 파이프라인 | 상호작용에서 문제를 관찰, 분석, 우선순위화해 실행 가능한 인사이트 생성 | 확장 가능한 데이터 처리, 자동화와 사람 감독의 결합, 선제적 리스크 탐지, 개선 사이클의 기반 | 데이터 품질 의존, 문제 전달 없이는 매우 새로운 이슈를 간과할 수 없음 | 실패 진단, 패턴 탐지, 개선 백로그 구축, 대규모나 복잡한 시스템에 적합 |
    | 실험 | 통제된 환경에서 변경을 검증, 효과 측정, 사전 위험 감소 | 데이터 기반, 리스크 최소화, 변형 비교 가능, 실제 조건에 적응 | 유의성에 충분한 데이터 필요, 리소스 집약적, 매우 고위험 상황에서는 게이트 없이 부적합 | 개선 검증, 점진적 롤아웃, 비교, 빠른 피드백이 필요한 동적 환경에 적합 |
    | 지속 학습 | 상호작용과 변화하는 요구에 기반한 동적 적응 내재화 | 실시간 적응성, 사용자 변화 대응, 견고성 강화, 개인화 지원 | 과적합/회귀 위험, 계산 비용 큼, 견고한 모니터링 필요 |  |

## 피드백 파이프라인

* 대규모로 운영되는 멀티 에이전트 시스템이 생성하는 방대한 볼륨과 복잡도의 데이터를 처리하려면 자동화된 피드백 파이프라인이 필수
    * 1차 분석선으로서 상호작용을 지속 모니터링, 실패 패턴 감지, 이슈 클러스터링으로 실행 가능한 인사이트 도출

* 자동 피드백 파이프라인 핵심 기능은 에이전트 워크플로 전반에서 반복되는 문제를 체계적으로 식별하는 것

* DSPy, APO 등 프레임워크가 사용하는 전형적인 자동 프롬프트 최적화 루프
    * 초기 프롬프트에서 타깃, 평가 모델을 거치고 데이터셋기반 점수에 따라 최적화 모델이 정련된 프롬프트를 생성

* 원시 관측 데이터를 반복적 개선으로 변환해 멀티 에이전트 시스템이 견고하고 적응적으로 유지되게 함

* SOC 분석가 에이전트 예시

In [ ]:
@tool
def lookup_threat_intel(indicator: str, type: str, **kwargs) -> str:
    """IP 주소, 파일 해시, URL 및 도메인에 대한 위협 인텔리전스를 조회합니다."""
    print(f"[TOOL] lookup_threat_intel(indicator={indicator}, type={type}, kwargs={kwargs})")
    log_to_loki("tool.lookup_threat_intel", f"indicator={indicator}, type={type}")
    return "threat_intel_retrieved"

@tool
def query_logs(query: str, log_index: str, **kwargs) -> str:
    """인증, 엔드포인트, 네트워크, 방화벽 및 DNS 시스템 전반에 걸쳐 보안 로그를 검색하고 분석합니다."""
    print(f"[TOOL] query_logs(query={query}, log_index={log_index}, kwargs={kwargs})")
    log_to_loki("tool.query_logs", f"query={query}, log_index={log_index}")
    return "log_query_executed"

@tool
def triage_incident(incident_id: str, decision: str, reason: str, **kwargs) -> str:
    """보안 사고를 실제 긍정(True Positive), 오탐지(False Positive) 또는 추가 조사를 위한 에스컬레이션으로 분류합니다."""
    print(f"[TOOL] triage_incident(incident_id={incident_id}, decision={decision}, reason={reason}, kwargs={kwargs})")
    log_to_loki("tool.triage_incident", f"incident_id={incident_id}, decision={decision}")
    return "incident_triaged"

@tool
def isolate_host(host_id: str, reason: str, **kwargs) -> str:
    """측면 이동을 방지하고 보안 사고를 억제하기 위해 손상된 호스트를 격리합니다."""
    print(f"[TOOL] isolate_host(host_id={host_id}, reason={reason}, kwargs={kwargs})")
    log_to_loki("tool.isolate_host", f"host_id={host_id}, reason={reason}")
    return "host_isolated"

@tool
def send_analyst_response(incident_id: str = None, message: str = None) -> str:
    """보안 분석, 사고 업데이트 또는 권장 사항을 이해 관계자에게 전송합니다."""
    print(f"[TOOL] send_analyst_response → {message}")
    log_to_loki("tool.send_analyst_response", f"incident_id={incident_id}, message={message}")
    return "analyst_response_sent"

TOOLS = [
    lookup_threat_intel, query_logs, triage_incident, isolate_host, send_analyst_response
]

* 자동화된 피드백 파이프라인은 대규모 멀티 에이전트 시스템이 생성하는 방대한 데이터의 볼륨과 복잡성을 다루는 데 필수

* 현대 피드백 도구의 강력한 능력 하나는 텍스트 기반 피드백을 시스템의 프롬프트, 스킬 파라미터, 추론 전략으로 직접 역전파하는 것
* 자동 파이프라인은 선제 최적화도 지원

* 자동 파이프라인의 오류
    * 문맥적 뉘앙스를 완전히 반영하거나 더 넓은 전략적 목표에 따라 개선의 우선순위를 정할 수는 없음

* 자동 피드백 파이프라인은 관찰, 클러스터링, 분석, 개선 제안을 반복하며 프롬프트, 도구, 추론 흐름 전반에 걸쳐 확장 가능한 자기 개선 루프를 만듦

### 자동화된 이슈 탐지와 근본 원인 분석

* 에이전틱 시스템의 복잡도가 증가하면 자동화된 이슈 탐지와 근본 원인 분석(RCA)이 필수적

* 자동 이슈 탐지는 규칙 기반 트리거, 이상 탐지 알고리즘, 통계적 클러스터링을 결합해 방대한 로그와 이벤트를 선별하며 다음의 패턴을 표시할 수 있음
    * 특정 스킬 또는 도구에서의 반복 실패
    * 에러율이나 응답 시간의 급증
    * 사용자 참여나 만족도 지표의 이상
    * 에이전트 버전이나 배포 환경 간 상이한 작동 방식

* 현대 피드백 파이프라인은 ML, 통계 기법을 이용해 서서히 나타나는 경향성도 포착
* 이슈가 탐지되면 RCA는 무엇이 실패했는지에 그치지 않고 왜 실패했는지를 물어봄

* RCA의 단계
    * 워크플로 추적
        * 실패에 이르기까지의 에이전트 결정, 도구 호출, 사용자 상호작용의 엔드 투 엔드 체인을 재구성
    * 결함 국소화
        * 오작동의 정확한 구성요소를 분리
    * 패턴 인식
        * 실패가 고립 사건인지 반복 추세의 일부인지 식별
        * 특정 사용자 코호트, 데이터 입력, 시스템 상태에 영향을 받을 수 있음
    * 영향 평가
        * 빈도와 심각도를 평가해 대응 우선순위를 정함

* RCA는 실패가 순수 기술적 원인만으로 발생하지 않음을 자주 보여줌
* 실행 가능한 RCA는 책임 소재를 가리는 데서 멈추지 않음
    * 프롬프트 및 도구 정제
    * 스킬 오케스트레이션 변경
    * 사용자 요구의 표현법과 전달 방식의 재고 등 의미 있는 시스템 개선 기회를 알 수 있음

* 자동화된 이슈 탐지와 RCA를 중심축으로 하는 견고한 피드백 파이프라인은 끝없는 트리아지에서 벗어나 모든 실패를 학습으로 전환하는 규율 있는 인사이트 중심 프로세스로 팀을 이동시킴

### 인간 개입 리뷰

* 자동 분석만으로는 충분하지 않은 경우가 여전히 많음
    * 모호한 사용자 의도
    * 윤리적 뉘앙스
    * 상충하는 목표
    * 새로운 엣지 케이스 관련 이슈  
    
    -> 인간의 직관, 도메인 전문성, 문맥적 판단이 필요

* 인간 개입 리뷰는 자동 탐지와 RCA를 보완하는 핵심 장치

* 인간 개입 리뷰 워크플로
    1. 입력 데이터가 에이전트를 통해 출력 후보로 생성
    2. 인간 리뷰와 수동 피드백을 거쳐 승인된 출력으로 이어지며 시스템 피드백 루프가이를 지원

* 인간 개입 리뷰는 가장 복잡하고 모호하거나 영향이 큰 시스템 이슈에 인간 판단을 투입하는 구조화된 보고 프로세스
    * 보고 기준의 예시
        * 명확한 기술적 설명 없이 지속되는 오류
        * 규제적, 윤리적 함의가 있는 워크플로의 이상
        * 고가치 또는 미션 크리티컬 작업의 실패
        * 자동 도구 간 상충하는 권고나 진단

* 인간과 AI의 의사결정 균형을 적절히 맞추려면 모델 확신도가 가장 낮거나 결과 영향이 가장 큰 사례를 우선 보고해야 함
* 고영향 사례의 경우 도메인별 심각도를 기준으로 영향을 평가해야 함

* 일반적인 리뷰 절차
    * 문맥 분석
        * 통제된 환경에서 실패나 이상을 재현해 사건의 순서와 의사결정 지점을 이해
    * 트레이스 점검
        * 로그, 트레이스, 의사결정 체인을 검사해 에이전트가 사용자 의도를 어떻게 해석하고 행동을 선택했는지 명확히 함
    * 영향 평가
        * 기술적 정확도와 UX를 모두 고려해 이슈의 범위와 심각도를 평가
    * 해결안 설계
        * 프롬프트 정제, 워크플로 재설계, 신규 스킬 개발, 사용자 대면 기능 변경 등 표적 개입을 제안

* 효과적인 인간 개입 리뷰 프로토콜은 문서화와 재현성을 중시
* 인간 개입 리뷰는 다양한 관점의 이점을 얻음
    * 프로덕트 매니저
        * 관찰된 실패가 더 깊은 사용자 요구 불일치를 반영하는지 밝힐 수 있음
    * UX 연구자
        * 자동 지표가 놓칠 수 있는 상호작용의 마찰 지점을 드러낼 수 있음
    * 데이터 사이언티스트
        * 다른 이들이 보지 못한 패턴이나 엣지 케이스 인지 가능
        
    -> 개선이 기술적 + 최종 사용자에게도 의미 있고 가치있도록 보장

* 자동화와 인간 감독의 균형을 통해 인간 개입 리뷰는 멀티 에이전트 시스템이 확장 가능하면서도 신뢰할 수 있도록 보장

### 프롬프트와 도구 정제

* 인사이트가 도출되면 그 다음은 표적 개선을 구현하는 일
* 시스템 정제를 위한 가장 직접적이고 영향력 있는 지렛대는 프롬프트의 설계와 외부 도구의 구성 및 호출  
    -> 프롬프트를 정제하는 것은 전체 성능을 높이는 매우 효율적인 방법이 될 수 있음

#### 프롬프트 정제

* 프롬프트는 사용자 의도와 에이전트 행동을 잇는 다리
* 프롬프트의 문구, 구조, 컨텍스트를 미세하게 바꾸는 것만으로도 에이전트의 해석, 추론, 출력이 극적으로 달라질 수 있음

* 피드백 루프의 문제
    * 지시가 모호해 응답이 일관성 없거나 무관해짐
    * 과도하게 광범위한 프롬프트가 할루시네이션이나 엉뚱한 출력 유발
    * 지나치게 경직되고 좁은 프롬프트가 현실 세계 변이에 일반화 실패
    * 작업 경계, 전달, 에러 처리에 대한 명확성 부족

* 정제는 분석에서 시작
* 빗나간 사례를 리뷰, 에이전트의 추론을 추적, 바람직하지 않은 결과에 기여한 프롬프트의 부분을 분리

* 개선 방법 예시
    * 명료성을 위한 재작성
        * 지시를 더 명확히 하고 모호성을 줄이며 기대하는 응답 형식을 지정
    * 예시 추가
        * 긍정/부정 예시를 프롬프트에 제공해 에이전트 추론을 고정
    * 작업 분해
        * 복잡한 다단계 지시를 더 작은 연속 프롬프트나 중간 추론 단계로 쪼갬
    * 컨텍스트 확장
        * 추가 컨텍스트, 제약, 관련 배경을 포함해 에이전트를 더 효과적으로 안내

* DSPy를 사용한 리액트 모듈 최적화 예시

In [ ]:
import dspy
dspy.configure(lm=dspy.LM("gpt-5-mini"))

def lookup_threat_intel(indicator: str) -> str:
    """모의: 인디케이터에 대한 위협 인텔리전스를 조회합니다."""
    return f"Mock intel for {indicator}: potentially malicious"

def query_logs(query: str) -> str:
    """모의: 보안 로그를 검색하고 분석합니다."""
    return f"Mock logs for '{query}': suspicious activity detected"

# 소수의 합성 테스트 케이스(경보 → 기대 응답)
# 실제로는 실제 로그에서 파생하거나 실패 사례에 주석을 달아 수집합니다.
# 더 나은 최적화를 위해 100개+를 목표로 하세요.
trainset = [
    dspy.Example(alert='''Suspicious login attempt from IP 203.0.113.45 to 
                 admin account.''',
                 response='''Lookup threat intel for IP, query logs for activity, 
                     triage as true positive, isolate host if malicious.''')
                     .with_inputs('alert'),
    dspy.Example(alert="Unusual file download from URL example.com/malware.exe.",
                 response='''Lookup threat intel for URL and hash, query logs 
                     for endpoint activity, triage as true positive, isolate 
                     host.''').with_inputs('alert'),
    dspy.Example(alert="High network traffic to domain suspicious-site.net.",
                 response='''Lookup threat intel for domain, query logs for 
                     network and firewall, triage as false positive if 
                     benign.''').with_inputs('alert'),
    dspy.Example(alert='''Alert: Potential phishing email with attachment 
                 hash abc123.''',
                 response='''Lookup threat intel for hash, query logs for email 
                     and endpoint, triage as true positive, send analyst 
                     response.''').with_inputs('alert'),
    dspy.Example(alert='''Anomaly in user behavior: multiple failed logins from 
                 new device.''',
                 response='''Query logs for authentication, lookup threat intel 
                     for device IP, triage as true positive if pattern matches 
                     attack.''').with_inputs('alert'),
]

# SOC 인시던트 처리를 위한 리액트 모듈 정의
react = dspy.ReAct("alert -> response", tools=[lookup_threat_intel, query_logs])

# 단순 지표를 사용하는 옵티마이저
# (예시를 위해 exact match 사용. 프로덕션에서는
# 시맨틱 유사도 같은 더 정교한 지표를 사용하세요)
tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", 
                  num_threads=24)
optimized_react = tp.compile(react, trainset=trainset)

* 고급 피드백 시스템에서는 관찰된 실패 패턴에 반응해 프롬프트 조정을 자동화할 수도 있음
    * 다만 회귀나 의도치 안흔 부작용을 막기 위해 모든 변경은 오프라인 테스트와 라이브 섀도 배포 모두에서 검증하는 것이 바람직함

#### 도구 정제

* 현대 에이전트 아키텍처에서 프롬프트만으로는 충분하지 않은 경우가 많음
* 에이전트는 다양한 도구에 점점 더 의존

* 피드백 파이프라인의 흔한 문제
    * 주어진 사용자 작업에 대해 잘못되었거나 비최적의 도구 선택
    * 도구 호출 파라미터 불일치나 잘못된 입력
    * 도구셋의 공백
    * 도구 체이닝 실패

* 도구 정제의 수준
    * 내부 로직 정제
        * 도구 내부의 프롬프트나 모델을 최적화해 데이터를 더 잘 처리, 분류하도록 함
    * 기능 확장
        * 최적화된 추론을 도입해 더 넓은 시나리오를 커버하도록 도구를 강화
    * 통합 개선
        * 도구가 에이전트의 요구에 부합하는 신뢰할 수 있고 실행 가능한 결과를 내도록 보장

* 확장된 DSPy 예시

In [ ]:
import dspy
dspy.configure(lm=dspy.LM("gpt-5-mini"))

# 위협 분류 작업을 위한 DSPy 시그니처 정의
class ThreatClassifier(dspy.Signature):
   """주어진 인디케이터(IP, URL, 해시 등)의 위협 수준을 
   'benign', 'suspicious', 'malicious' 중 하나로 분류합니다."""
   indicator: str = dspy.InputField(desc="IP 주소, URL, 파일 해시 등 분류할 인디케이터.")
   threat_level: str = dspy.OutputField(desc="분류된 위협 수준: 'benign', 'suspicious', 또는 'malicious'.")

# 사려 있는 분류를 위한 ChainOfThought 기반 DSPy 모듈
class ThreatClassificationModule(dspy.Module):
   def __init__(self):
       super().__init__()
       self.classify = dspy.ChainOfThought(ThreatClassifier)
  
   def forward(self, indicator):
       return self.classify(indicator=indicator)


# 평가 지표(위협 수준의 exact match.
# 프로덕션에서는 시맨틱 매치나 커스텀 스코어러 권장)
def threat_match_metric(example, pred, trace=None):
   return example.threat_level.lower() == pred.threat_level.lower()

# 모듈 최적화(다양한 사례 처리를 위한 내부 프롬프트를 정제)
optimizer = dspy.BootstrapFewShotWithRandomSearch(metric=threat_match_metric, 
    max_bootstrapped_demos=4, max_labeled_demos=4)
optimized_module = optimizer.compile(ThreatClassificationModule(), 
                                     trainset=trainset)

# 도구에서의 사용 예: 최적화 후 classify_threat에 사용
def classify_threat(indicator: str) -> str:
   """최적화된 DSPy 모듈을 사용해 위협 수준을 분류합니다."""
   prediction = optimized_module(indicator=indicator)
   return prediction.threat_level

* 각 프롬프트 또는 도구 정제는 명확한 근거를 문서화해야 함
    * 어떤 문제가 관찰되었는지
    * 어떤 변경을 했는지
    * 효과를 어떻게 측정할 것인지

    -> 추적성과 재현성을 보장, 무엇이 왜 효과적인지에 대한 지식 베이스를 후속 팀에 제공

* 정제는 오프라인 평가와 통제된 라이브 실험을 모두 사용해 반복적으로 검증해야 함
* 배포 후 모니터링은 특히 중요

* 프롬프트와 도구 정제는 에이전틱 시스템에서 진전을 이끄는 손에 잡히는 수단
* 인사이트를 행동으로 연결, 사려 깊게 반복으로 팀은 모든 실패나 마찰 지점을 더 견고하고 반응성이 뛰어나며 유능한 AI로 나아가는 기회로 바꿀 수 있음

### 개선 항목 집계와 우선순위화

* 에이전틱 시스템의 복잡도와 규모가 커지면 피드백 파이프라인과 인간 개입 리뷰에서 도출되는 실행 가능한 인사이트의 흐름도 함께 커짐

* 첫 단계는 집계
    * 여러 출처의 인사이트를 통합해 접근 가능한 단일 뷰로 만드는 일
    * 핵심 실천
        * 중복 제거
            * 유사 이슈를 클러스터링해 노력의 분절을 방지
        * 태깅과 분류
            * 근본 원인, 영향 워크플로, 사용자 임팩트, 시스템 컴포넌트 기준으로 라벨링해 정렬, 필터링을 쉽게 함
        * 컨텍스트 연결
            * 각 개선 항목에 로그, 트레이스, 사용자 리포트, RCA 문서를 첨부해 효율적 트리아지와 조치를 가능하게 함

* 단일 백로그가 준비되면 다음 과제는 우선순위화
    * 효과적인 우선순위화의 고려사항
        * 빈도
        * 심각도/영향
        * 실행 용이성
        * 전략 정렬
        * 재발 가능성과 리스크

* 집계와 우선순위화의 규율은 방대한 피드백 스트림을 명확하고 실행 가능한 로드맵으로 바꿈

## 실험

* 실험은 멀티 에이전트 시스템에서 안전한 진전을 이끄는 엔진
* 인사이트와 배포를 잇는 다리로서 변경을 검증, 실제 효과를 측정, 광범위한 업데이트 전에 위험을 완화할 수 있음
* 에이전틱 아키텍처는 사소한 수정도 광범위하고 때로는 예측하기 어려운 파급을 나흘 수 있음  
    -> 제대로 된 실험 체계가 없으면 변경 사항이 오히려 이전보다 나빠지는 결과로 이어질 수 있음

* 잘 설계된 실험 프로세스는 변화를 위한 체계적이고 점진적인 경로를 제공
    * 아이디어 -> 프로덕션이 아닌 실제 환겨을 가깝게 모사한 통제된 환경에서 변경을 도입하고 평가
    * 보통 스테이징, 릴리스 후보(RC) 환경에서 시작해 라이브 사용자에게 영향 없이 문제를 조기에 포착하는 것이 표준 모범 사례
    * 이후 섀도 배포, 카나리 롤아웃, 롤링 업데이트, 블루/그린 배포 등 적용 가능

    -> 데이터 주도 의사 결정의 토대 마련

### 섀도 배포

* 사용자를 노출하지 않고도 실제 조건에서 시스템 변경을 검증하는 강력한 방법
* 나란히 비교함으로써 새로운 또는 업데이트된 에이전트 로직이 실제 운영 부하에서 어떻게 작동하는지 관찰, 측정, 진단할 수 있음

* 이점
    * 현실적 검증
    * 안전한 탐색
    * 엣지 케이스 발견
    * 보완 전략과의 통합

### A/B 테스트

* 라이브 트래픽을 대조군(A)와 실험군(B)으로 분할해 비교
* 사용자는 둘 중 하나와 상호작용하고 협업 에이전트 스웜에서의 작업 성공률이나 응답의 할루시네이션 감소 같은 지표에서 정량적 우위를 도출
* 측정 가능한 미세 조정에 특히 강함

* 강점
    * 현실 적합성
    * 직접 비교
    * 통계적 엄밀성

* 다만 대화 기록이나 지속되는 사용자 컨텍스트처럼 장기 상호작용 상태를 저장하는 경우 A/B 테스트가 어려워질 수 있음  
    -> 사용자가 다른 세션에서는 다른 버전에 할당되면 불일치를 겪음
    -> 고정 배정을 적용하거나 세션 수준에서 테스트를 수행 하거나 싱태 관리를 격리해 교차 오염 방지

### 베이지안 밴딧

* 실험 도중에도 학습해 승자 쪽으로 사용자를 점차 이동시킴
* 예측 불가능한 환경에서 개선 속도를 높이기 위해 탐색과 활용을 동적으로 균형 잡는 적응형 실험의 핵심 기법

* 주요 장점
    * 반응성
    * 효율성
    * 확장성

* 적응형 실험의 요구사항
    * 지표 정합성
    * 신중한 초기화
    * 면밀한 감독

## 지속 학습

* 에이전틱 시스템이 실제 상호작용, 피드백, 변화하는 사용자 요구에 따라 시간이 지남에 따라 적응, 개선, 최적화되도록 설계하는 단계
* 지속적으로 학습하는 에이전트는 새로운 데이터를 흡수하고 작동 과정을 정제하며 추론 전략을 동적으로 업데이트하도록 설계됨

* 지속 학습은 두 가지 핵심 메커니즘을 포괄
    * 인컨텍스트 학습
    * 온라인 학습

    -> 세션 내 실시간 미세 조정부터 워크플로 전반의 점진적인 업데이트까지 다양한 규모의 개선을 가능하게 함

### 인컨텍스트 학습

* 파운데이션 모델 기반 시스템에서 가장 즉각적이고 유연한 적응 수단
* 단일 세션 안에서 에이전트가 작동 과정을 동적으로 바꾸도록 함
    * 프롬프트에 예시, 중간 추론 단계, 문맥 신호를 직접 포함해 에이전트에 즉석에서 새로운 행동을 가르침
    * 런타임에 적응하도록 함

* 주요 강점
    * 사용자 맞춤 적응
    * 실시간 피드백 반영
    * 유도된 추론

* 한계
    * 세션 안에서 이루어진 변경은 휘발성  
        -> 가치 있는 인사이트의 보존을 위해선 인컨텍스트 전략을 더 영속적인 메커니즘으로 승격해야 함

### 오프라인 재학습

* 피드백 파이프라인과 실험에서 축적된 데이터를 토대로, 이에진트 시스템에 지속적 개선을 내재화하는 정기적이며 구조화된 접근
* 사용자 질의, 에이전트 출력, 라벨링된 결과 같은 상호작용 데이터를 배치로 수집해, 비프로덕션 환경에서 프롬프트와 도구를 업데이트하거나 파운데이션 모델을 파인튜닝

* 주요 강점
    * 지속성
    * 확장성
    * 리스크 완화

* 과거 데이터에 과적합하거나 새로운 추세를 간과하지 않도록 세심한 관리가 필요함
* 피드백과 실험과 결합하면 오프라인 재학습은 인사이트를 지속 가능한 향상으로 번역해 개선 루프를 완성